# Tutorial 5: Depairing Channels

This tutorial demonstrates the `supermag.depairing` module, which computes
dimensionless pair-breaking parameters from physical laboratory inputs.

Pair-breaking (depairing) channels suppress superconductivity. The total
depairing parameter is the sum of four contributions:

$$\lambda_{\mathrm{dep}} = \lambda_{\mathrm{AG}} + \lambda_Z + \lambda_{\mathrm{orb}} + \lambda_{\mathrm{SO}}$$

Each channel arises from a different physical mechanism:

| Channel | Physical origin | Equation |
|:--------|:----------------|:---------|
| $\lambda_{\mathrm{AG}}$ | Spin-flip (Abrikosov–Gor'kov) | $\Gamma_s / 2k_BT$ |
| $\lambda_Z$ | Zeeman (Pauli paramagnetic) | $(\mu_B H)^2 / (2\pi k_BT)^2$ |
| $\lambda_{\mathrm{orb}}$ | Orbital (diamagnetic) | $D(eH)^2 d^2 / 3\hbar^2 2\pi k_BT$ |
| $\lambda_{\mathrm{SO}}$ | Spin–orbit coupling | $\Gamma_{\mathrm{so}} / 2k_BT$ |

In [ ]:
import sys
sys.path.insert(0, r"c:\Users\seans\Documents\GitHub\SUPERMag-Simulation-Suite\python")

import numpy as np
import matplotlib.pyplot as plt
import supermag

print(f"SUPERMag v{supermag.__version__}")
print(f"Depairing functions: {[x for x in dir(supermag) if 'depairing' in x]}")

## 1. Individual Depairing Channels

Each function takes physical lab inputs and returns a dimensionless
pair-breaking parameter. Let’s compute all four channels for a
realistic Nb thin film at $T = 5\,$K.

In [ ]:
T = 5.0  # K

# Spin-flip (AG): scattering rate in meV
lam_ag = supermag.depairing_ag(gamma_s_meV=0.05, T_kelvin=T)

# Zeeman: applied field in Tesla
lam_z = supermag.depairing_zeeman(H_tesla=2.0, T_kelvin=T)

# Orbital (perpendicular field): D in nm²/ps, thickness in nm
lam_orb = supermag.depairing_orbital_perp(
    D_nm2ps=4.0, H_tesla=2.0, thickness_nm=50.0, T_kelvin=T)

# Spin-orbit: scattering rate in meV
lam_so = supermag.depairing_soc(Gamma_so_meV=0.01, T_kelvin=T)

print(f"λ_AG   = {lam_ag:.6f}")
print(f"λ_Z    = {lam_z:.6f}")
print(f"λ_orb  = {lam_orb:.6f}")
print(f"λ_SO   = {lam_so:.6f}")
print(f"λ_tot  = {lam_ag + lam_z + lam_orb + lam_so:.6f}")

## 2. Temperature Dependence

All channels depend on temperature. The AG and spin–orbit channels
scale as $1/T$, the Zeeman channel as $1/T^2$, and the orbital
channel as $1/T$. Let’s plot them together.

In [ ]:
T_arr = np.linspace(1.0, 9.0, 200)

ag_arr  = [supermag.depairing_ag(0.05, T) for T in T_arr]
z_arr   = [supermag.depairing_zeeman(2.0, T) for T in T_arr]
orb_arr = [supermag.depairing_orbital_perp(4.0, 2.0, 50.0, T) for T in T_arr]
so_arr  = [supermag.depairing_soc(0.01, T) for T in T_arr]
total   = np.array(ag_arr) + np.array(z_arr) + np.array(orb_arr) + np.array(so_arr)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_arr, ag_arr,  label=r'$\lambda_{\mathrm{AG}}$', lw=2)
ax.plot(T_arr, z_arr,   label=r'$\lambda_Z$', lw=2)
ax.plot(T_arr, orb_arr, label=r'$\lambda_{\mathrm{orb}}$', lw=2)
ax.plot(T_arr, so_arr,  label=r'$\lambda_{\mathrm{SO}}$', lw=2)
ax.plot(T_arr, total,   'k--', label=r'$\lambda_{\mathrm{total}}$', lw=2, alpha=0.7)
ax.set_xlabel('Temperature (K)', fontsize=13)
ax.set_ylabel(r'Depairing parameter $\lambda$', fontsize=13)
ax.set_title('Depairing channels vs. temperature', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Magnetic Field Scan

The Zeeman and orbital channels increase with the applied field.
The orbital contribution dominates for thick films or high fields.

In [ ]:
H_arr = np.linspace(0.0, 5.0, 300)
T_fixed = 5.0

z_vs_H   = [supermag.depairing_zeeman(H, T_fixed) for H in H_arr]
orb_perp = [supermag.depairing_orbital_perp(4.0, H, 50.0, T_fixed) for H in H_arr]
orb_par  = [supermag.depairing_orbital_par(4.0, H, 50.0, T_fixed) for H in H_arr]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(H_arr, z_vs_H,   label=r'$\lambda_Z$ (Zeeman)', lw=2)
ax.plot(H_arr, orb_perp, label=r'$\lambda_{\mathrm{orb},\perp}$', lw=2)
ax.plot(H_arr, orb_par,  label=r'$\lambda_{\mathrm{orb},\parallel}$', lw=2, ls='--')
ax.set_xlabel('Magnetic field $H$ (T)', fontsize=13)
ax.set_ylabel(r'Depairing parameter $\lambda$', fontsize=13)
ax.set_title('Field-dependent depairing at $T = 5$ K, $d = 50$ nm', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Perpendicular vs. Parallel Orbital Depairing

The orbital channel depends on field orientation. For a thin film of
thickness $d$:

$$\lambda_{\mathrm{orb},\perp} = 4 \, \lambda_{\mathrm{orb},\parallel}$$

This factor of 4 comes from the geometry of the vector potential.

In [ ]:
# Verify the ratio analytically
lam_perp = supermag.depairing_orbital_perp(4.0, 2.0, 50.0, 5.0)
lam_par  = supermag.depairing_orbital_par(4.0, 2.0, 50.0, 5.0)

print(f"λ_orb,⊥  = {lam_perp:.8f}")
print(f"λ_orb,∥  = {lam_par:.8f}")
print(f"Ratio ⊥/∥ = {lam_perp / lam_par:.1f}  (expected: 4.0)")

## 5. Composite Depairing: `depairing_from_physical()`

For convenience, `depairing_from_physical()` computes all four channels
in a single call and returns a dictionary.

In [ ]:
channels = supermag.depairing_from_physical(
    gamma_s_meV=0.05,
    H_tesla=2.0,
    D_nm2ps=4.0,
    thickness_nm=50.0,
    Gamma_so_meV=0.01,
    T_kelvin=5.0,
)

print("Depairing channels:")
for name, val in channels.items():
    print(f"  {name:12s} = {val:.6f}")
print(f"  {'total':12s} = {sum(channels.values()):.6f}")

## 6. Channel Bar Chart

Which depairing mechanism dominates? Let’s compare channels for
three different experimental conditions.

In [ ]:
conditions = {
    'Low field\n(0.5 T)': dict(gamma_s_meV=0.05, H_tesla=0.5,
                               D_nm2ps=4.0, thickness_nm=50.0,
                               Gamma_so_meV=0.01, T_kelvin=5.0),
    'High field\n(5 T)':  dict(gamma_s_meV=0.05, H_tesla=5.0,
                               D_nm2ps=4.0, thickness_nm=50.0,
                               Gamma_so_meV=0.01, T_kelvin=5.0),
    'Dirty film\n(high $\\Gamma_s$)': dict(gamma_s_meV=0.5, H_tesla=1.0,
                               D_nm2ps=4.0, thickness_nm=50.0,
                               Gamma_so_meV=0.1, T_kelvin=5.0),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']

for ax, (label, kw) in zip(axes, conditions.items()):
    ch = supermag.depairing_from_physical(**kw)
    names = list(ch.keys())
    vals = list(ch.values())
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel(r'$\lambda$', fontsize=13)
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Depairing channel comparison', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 7. Effect on $T_c$ Suppression

Depairing channels can be passed to `critical_temperature()` to see
their effect on the $T_c(d_F)$ curve.

In [ ]:
nb = supermag.get_material('Nb')
fe = supermag.get_material('Fe')

d_F = np.linspace(0.5, 15, 200)

# Without depairing
Tc_bare = supermag.critical_temperature(
    Tc0=nb['Tc'], d_S=50.0, d_F_array=d_F,
    E_ex=fe['E_ex'], xi_S=nb['xi_S'], xi_F=fe['xi_F'])

# With depairing channels
dp = supermag.depairing_from_physical(
    gamma_s_meV=0.05, H_tesla=1.0, D_nm2ps=4.0,
    thickness_nm=50.0, Gamma_so_meV=0.01, T_kelvin=5.0)

Tc_dep = supermag.critical_temperature(
    Tc0=nb['Tc'], d_S=50.0, d_F_array=d_F,
    E_ex=fe['E_ex'], xi_S=nb['xi_S'], xi_F=fe['xi_F'],
    depairing=dp)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d_F, Tc_bare, 'b-', lw=2, label='No depairing')
ax.plot(d_F, Tc_dep, 'r--', lw=2, label='With depairing')
ax.axhline(nb['Tc'], ls=':', color='gray', lw=0.8)
ax.set_xlabel(r'$d_F$ (nm)', fontsize=13)
ax.set_ylabel(r'$T_c$ (K)', fontsize=13)
ax.set_title(r'$T_c(d_F)$: effect of depairing channels', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Max Tc suppression from depairing: {(Tc_bare - Tc_dep).max():.3f} K")

## Summary

| Function | Purpose | Key inputs |
|:---------|:--------|:-----------|
| `depairing_ag()` | Spin-flip (AG) channel | $\Gamma_s$, $T$ |
| `depairing_zeeman()` | Zeeman channel | $H$, $T$ |
| `depairing_orbital_perp()` | Orbital channel ($\perp$) | $D$, $H$, $d$, $T$ |
| `depairing_orbital_par()` | Orbital channel ($\parallel$) | $D$, $H$, $d$, $T$ |
| `depairing_soc()` | Spin–orbit channel | $\Gamma_{\mathrm{so}}$, $T$ |
| `depairing_from_physical()` | All channels at once | all of the above |

**Next steps:**
- Use the depairing dict with `critical_temperature()` and `tc_parameter_sweep()`
- Fit depairing-modified models to experimental data — see
  [06_fitting.ipynb](06_fitting.ipynb)